# Extension: Joint EO + Feature-Space Lipschitz Post-Processor

This notebook implements and evaluates a novel joint post-processor that simultaneously enforces:
1. **Equalized Odds** (equal FPR and FNR across racial groups)
2. **Feature-space Lipschitz constraint**: |p_i - p_j| ≤ L · d(x_i, x_j) for all KNN-connected pairs

The joint constraint is posed as a Quadratic Program (QP) solved with `cvxpy` + OSQP.

**Extension hypotheses:**
- **H_ext1**: What is the minimum L required for feasibility at exact EO (epsilon=0)?
- **H_ext2**: Does the joint constraint strictly dominate either constraint alone on all metrics?
- **H_ext3**: What accuracy cost does joint fairness impose relative to Hardt EO alone?

## 1. Setup

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src/ to path (mirrors run_pipeline.ipynb setup)
notebook_dir = os.path.abspath('')
project_root = os.path.dirname(notebook_dir)
src_path = os.path.join(project_root, 'src')
for p in [src_path, os.path.join(os.getcwd(), 'src'), 'src', '../src']:
    p = os.path.abspath(p)
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
})

from config import RANDOM_SEED, RESULTS_DIR
from extension.joint_postprocessor import (
    solve_joint, binarize, solve_joint_grid,
    L_GRID, EPSILON_GRID_EXT, EXT_RESULTS_DIR,
)
from extension.feasibility import (
    compute_feasibility_frontier, plot_feasibility_heatmap, plot_three_way_frontier,
    EXT_FIGURES_DIR,
)
from extension.comparison import compare_all_methods, plot_comparison_radar

os.makedirs(EXT_RESULTS_DIR, exist_ok=True)
os.makedirs(EXT_FIGURES_DIR, exist_ok=True)
print('Environment ready.')
print('Results directory:', EXT_RESULTS_DIR)
print('Figures directory:', EXT_FIGURES_DIR)

In [ ]:
# Load y and race for the test set by re-running the deterministic data pipeline
# (compas_clean.csv was saved without the pandas index, so we reconstruct it)
import data_prep as dp

df_full = dp.load_and_clean()  # reads data/compas_raw.csv, deterministic

# Use the saved test indices (same split as the original pipeline)
test_idx_arr = pd.read_csv(os.path.join(RESULTS_DIR, 'test_idx.csv'))['idx'].values

df_test = df_full.loc[test_idx_arr]
y_test   = df_test['two_year_recid'].values.astype(int)
race_test = df_test['race'].values.astype(int)
ids      = test_idx_arr  # original pandas row indices (used as IDs throughout)

print(f'Test set: n={len(ids)}')
print(f'Black: {race_test.sum()}  White: {(race_test==0).sum()}')
print(f'Recid rate: {y_test.mean():.3f}')

In [ ]:
# Load saved baseline scores, EO decisions, and KNN pairs
scores_df = pd.read_csv(os.path.join(RESULTS_DIR, 'baseline_scores.csv'), index_col=0)
s = scores_df.loc[ids, 'score'].values.astype(float)

pairs_df = pd.read_csv(os.path.join(RESULTS_DIR, 'knn_pairs_euclidean.csv'))

print(f'Baseline scores loaded: n={len(s)}')
print(f'KNN pairs: {len(pairs_df)} edges')
print(pairs_df.head(3))

## 2. Single Solve Demo

Run `solve_joint` at L=1.0, epsilon=0.05 as a sanity check before the full grid.

In [ ]:
demo_result = solve_joint(
    s, y_test, race_test, pairs_df,
    L=1.0, epsilon=0.05, ids=ids, verbose=False,
)

print('=== Single Solve Demo: L=1.0, epsilon=0.05 ===')
print(f"Status           : {demo_result['status']}")
print(f"Objective value  : {demo_result['obj_value']:.4f}")
print(f"FPR gap achieved : {demo_result['fpr_gap']:.4f}  (limit={0.05})")
print(f"FNR gap achieved : {demo_result['fnr_gap']:.4f}  (limit={0.05})")
print(f"Lipschitz viol.  : {demo_result['lipschitz_violations']}")

if demo_result['p_opt'] is not None:
    p_demo = demo_result['p_opt']
    d_demo = binarize(p_demo)
    acc = (d_demo == y_test).mean()
    print(f"Accuracy         : {acc:.4f}")
    print(f"Mean p_opt       : {p_demo.mean():.4f}  (cf. mean s={s.mean():.4f})")

In [ ]:
# Plot score distribution: baseline s vs joint p_opt
if demo_result['p_opt'] is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    for ax, vals, title in zip(
        axes,
        [s, demo_result['p_opt']],
        ['Baseline scores s', 'Joint p_opt (L=1.0, ε=0.05)'],
    ):
        ax.hist(vals[race_test == 1], bins=30, alpha=0.6, color='#EE7733', label='Black')
        ax.hist(vals[race_test == 0], bins=30, alpha=0.6, color='#0077BB', label='White')
        ax.set_title(title)
        ax.set_xlabel('Probability')
        ax.set_ylabel('Count')
        ax.legend()

    fig.suptitle('Score Distributions: Baseline vs Joint Post-Processor', fontsize=12)
    fig.tight_layout()
    fig.savefig(EXT_FIGURES_DIR + 'demo_score_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

## 3. Feasibility Grid

Sweep all (L, epsilon) combinations. The infeasible region is expected at small L + small epsilon.

In [ ]:
print(f'Grid: {len(L_GRID)} L values × {len(EPSILON_GRID_EXT)} epsilon values = {len(L_GRID)*len(EPSILON_GRID_EXT)} cells')
print(f'L_GRID         = {L_GRID}')
print(f'EPSILON_GRID   = {EPSILON_GRID_EXT}')
print('Running grid (monotonicity pruning applied — infeasible cells skip smaller L)...')

grid_df = solve_joint_grid(
    s, y_test, race_test, pairs_df,
    L_grid=L_GRID,
    epsilon_grid=EPSILON_GRID_EXT,
    ids=ids,
)

n_feasible = (grid_df['status'].isin(['optimal', 'optimal_inaccurate'])).sum()
print(f'\nGrid complete: {n_feasible}/{len(grid_df)} cells feasible')
print(grid_df.to_string())

In [ ]:
# Feasibility heatmap
heatmap_path = EXT_FIGURES_DIR + 'feasibility_heatmap.png'
plot_feasibility_heatmap(grid_df, heatmap_path)

from IPython.display import Image
Image(heatmap_path)

## 4. Feasibility Frontier

For each epsilon, the minimum L at which the QP is feasible.

In [ ]:
frontier_df = compute_feasibility_frontier(grid_df)
print('Feasibility frontier:')
print(frontier_df.to_string(index=False))

In [ ]:
# Plot L_min_feasible vs epsilon
frontier_valid = frontier_df.dropna(subset=['L_min_feasible'])

fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(
    frontier_valid['epsilon'], frontier_valid['L_min_feasible'],
    'o-', color='#009988', lw=2, ms=8, label='L_min feasible',
)
ax1.set_xlabel('Epsilon (EO tolerance)', fontsize=12)
ax1.set_ylabel('Minimum feasible L', color='#009988', fontsize=12)
ax1.tick_params(axis='y', labelcolor='#009988')

ax2 = ax1.twinx()
ax2.plot(
    frontier_valid['epsilon'], frontier_valid['inconsistency_rate_at_Lmin'],
    's--', color='#EE7733', lw=2, ms=7, label='Inconsistency at L_min',
)
ax2.set_ylabel('Inconsistency rate at L_min', color='#EE7733', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#EE7733')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
ax1.set_title('Feasibility Frontier: L_min vs Epsilon', fontsize=13)

fig.tight_layout()
fig.savefig(EXT_FIGURES_DIR + 'feasibility_frontier.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Three-way trade-off plot
three_way_path = EXT_FIGURES_DIR + 'three_way_frontier.png'
plot_three_way_frontier(grid_df, three_way_path)

Image(three_way_path)

## 5. Operating Point Selection

We choose (L_star, epsilon_star) on or near the feasibility frontier, balancing:
- **Group fairness**: small epsilon (strict EO)
- **Individual fairness**: small L (strict Lipschitz)
- **Accuracy**: not too far from baseline

In [ ]:
# Select operating point: epsilon=0.05 (moderate EO strictness) and L = L_min at that epsilon
# Rationale: epsilon=0.05 is a standard tolerance used in the original H3 sweep.
# L_star is the minimum L at which the QP is feasible for this epsilon — the tightest
# individual fairness constraint that still permits a valid solution.

TARGET_EPS = 0.05

row_target = frontier_df[frontier_df['epsilon'] == TARGET_EPS].iloc[0]
L_star = float(row_target['L_min_feasible']) if pd.notna(row_target['L_min_feasible']) else 1.0
epsilon_star = TARGET_EPS

print(f'Selected operating point:')
print(f'  L_star       = {L_star}')
print(f'  epsilon_star = {epsilon_star}')
print()
print('Justification:')
print('  epsilon=0.05 matches the strictness used in the original H3 analysis,')
print('  enabling direct comparison with the Hardt EO baseline.')
print('  L_star is the smallest feasible L at this epsilon — the Pareto-optimal')
print('  point that maximises individual fairness while satisfying group fairness.')

# Show the grid cell at this operating point
op_cell = grid_df[(grid_df['L'] == L_star) & (grid_df['epsilon'] == epsilon_star)]
if not op_cell.empty:
    print('\nGrid cell at operating point:')
    print(op_cell.to_string(index=False))

## 6. Three-Way Method Comparison

In [ ]:
comparison_df = compare_all_methods(
    s, y_test, race_test, pairs_df,
    L_star=L_star,
    epsilon_star=epsilon_star,
    ids=ids,
)

print('=== Method Comparison ===')
print(comparison_df.set_index('method').T.to_string())

In [ ]:
radar_path = EXT_FIGURES_DIR + 'radar_comparison.png'
plot_comparison_radar(comparison_df, radar_path)

Image(radar_path)

## 7. Key Findings

In [ ]:
# H_ext1: Minimum L required for exact EO (epsilon=0)
row_eps0 = frontier_df[frontier_df['epsilon'] == 0.0].iloc[0]
L_min_exact_eo = row_eps0['L_min_feasible']

print('=== H_ext1: Minimum L for exact EO (epsilon=0) ===')
if pd.notna(L_min_exact_eo):
    print(f'  L_min = {L_min_exact_eo}')
    print(f'  Interpretation: to simultaneously achieve exact EO AND the Lipschitz')
    print(f'  constraint, the Lipschitz constant must be at least {L_min_exact_eo}.')
    print(f'  Values below this make the joint QP infeasible.')
else:
    print('  Infeasible at all tested L values for epsilon=0.')
    print('  Joint exact EO + strict Lipschitz is incompatible in this dataset.')

In [ ]:
# H_ext2: Does Joint dominate either method alone?
print('=== H_ext2: Does Joint EO+Lip dominate Hardt EO and Petersen IF-only? ===')
print()

joint_row   = comparison_df[comparison_df['method'] == 'Joint EO+Lip'].iloc[0]
eo_row      = comparison_df[comparison_df['method'] == 'Hardt EO'].iloc[0]
petersen_row = comparison_df[comparison_df['method'] == 'Petersen IF-only'].iloc[0]

print('vs Hardt EO:')
print(f'  FPR gap  : Joint={joint_row["fpr_gap"]:.4f}  EO={eo_row["fpr_gap"]:.4f}  (lower=better)')
print(f'  IF rate  : Joint={joint_row["inconsistency_rate"]:.4f}  EO={eo_row["inconsistency_rate"]:.4f}  (lower=better)')
print(f'  Accuracy : Joint={joint_row["accuracy"]:.4f}  EO={eo_row["accuracy"]:.4f}')

print()
print('vs Petersen IF-only:')
print(f'  FPR gap  : Joint={joint_row["fpr_gap"]:.4f}  Petersen={petersen_row["fpr_gap"]:.4f}')
print(f'  IF rate  : Joint={joint_row["inconsistency_rate"]:.4f}  Petersen={petersen_row["inconsistency_rate"]:.4f}')
print(f'  Accuracy : Joint={joint_row["accuracy"]:.4f}  Petersen={petersen_row["accuracy"]:.4f}')

joint_dominates_eo = (
    joint_row['fpr_gap'] <= eo_row['fpr_gap'] and
    joint_row['inconsistency_rate'] <= eo_row['inconsistency_rate']
)
joint_dominates_pet = (
    joint_row['fpr_gap'] <= petersen_row['fpr_gap'] and
    joint_row['inconsistency_rate'] <= petersen_row['inconsistency_rate']
)

print()
print(f'Joint strictly dominates Hardt EO   : {joint_dominates_eo}')
print(f'Joint strictly dominates Petersen IF : {joint_dominates_pet}')

In [ ]:
# H_ext3: Accuracy cost of joint fairness
print('=== H_ext3: Accuracy cost of joint fairness ===')

baseline_acc = (binarize(s) == y_test).mean()
joint_acc    = joint_row['accuracy']
eo_acc       = eo_row['accuracy']

print(f'  Baseline accuracy          : {baseline_acc:.4f}')
print(f'  Hardt EO accuracy          : {eo_acc:.4f}  (delta={eo_acc - baseline_acc:+.4f})')
print(f'  Joint EO+Lip accuracy      : {joint_acc:.4f}  (delta={joint_acc - baseline_acc:+.4f})')
print(f'  Joint vs EO accuracy delta : {joint_acc - eo_acc:+.4f}')

In [ ]:
# Summary table
print('=== Full Results Summary ===')
display(comparison_df.set_index('method'))

print('\n=== Feasibility Frontier ===')
display(frontier_df)

print('\nAll results saved to:', EXT_RESULTS_DIR)
print('All figures saved to:', EXT_FIGURES_DIR)